# Unit 3, Lecture 1: The framework landscape

For two units you built agents by hand. Now you meet the real frameworks, and
the point of this notebook is that they are **not magic**: every framework
concept maps onto something you already wrote.

The syllabus names Semantic Kernel and AutoGen. They merged into **Microsoft
Agent Framework 1.0** in April 2026, so we learn the lineage and build on the
successor. This is what you would use in a job today.

**This unit runs against the real packages.** Install them once:

```bash
pip install agent-framework semantic-kernel
```

## The landscape, as data you can read

In [ ]:
from cse476.frameworks import describe_landscape, framework_by_status

for f in describe_landscape():
    print(f"{f.name:28} [{f.status}]")
    print(f"    lineage:  {f.lineage}")
    print(f"    use when: {f.when_to_use}")
    print()

In [ ]:
print("current, build new things on these:", framework_by_status("current"))
print("maintenance, do not start here:     ", framework_by_status("maintenance"))
print("community fork:                     ", framework_by_status("community fork"))

That table is the interview answer. An interviewer does not ask "do you know
framework X". They ask "how would you choose", and being able to name the current
options, the maintenance-mode ones, and why they are in each state is what a
strong candidate sounds like.

## Every framework word maps to something you built

Before we run any framework code, look at what its vocabulary actually means. The
left column is framework jargon. The right column is code you already wrote.

In [ ]:
from cse476.frameworks import CONCEPT_MAP

for framework_word, what_you_built in CONCEPT_MAP.items():
    print(f"{framework_word:22} ->  {what_you_built}")

Nothing on the left is a new idea. It is your Unit 1 and Unit 2 work, given
industrial names. That is what a framework is: convenient, battle-tested
packaging of ideas you now understand from the inside.

## The same agent, three ways

Now prove it. The same "answer in one sentence" agent, built by hand, on Agent
Framework, and on Semantic Kernel. **These make live model calls, so you need a
lane configured** (`.env` with a free lane like `PROVIDER=groq` is enough).

First, load your lane so the frameworks can use the same credentials.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# the frameworks read whatever lane is active in your .env (groq, local, or foundry)
from cse476.lanes import get_connection, describe
print(describe())   # shows the active lane and model
base_url, api_key, model = get_connection()


### By hand (Unit 1), for reference

In [ ]:
from cse476.lanes import get_client, MODEL

client = get_client()
reply = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant. Answer in one sentence."},
        {"role": "user", "content": "What is an AI agent?"},
    ],
).choices[0].message.content
print("BY HAND:      ", reply)

### On Microsoft Agent Framework (the successor)

Watch how little there is: a chat client, turned into an agent with
instructions, then run. The client takes the **same** model, key, and base url
as your lane.

In [ ]:
from agent_framework.openai import OpenAIChatClient

# lane-aware: base_url, api_key, model all come from get_connection above,
# so this runs on whatever provider your .env selects. No provider named here.
af_client = OpenAIChatClient(model=model, api_key=api_key, base_url=base_url)
agent = af_client.as_agent(instructions="You are a concise assistant. Answer in one sentence.")

result = await agent.run("What is an AI agent?")
print("AGENT FRAMEWORK:", result)


`OpenAIChatClient` is your `get_client`. `as_agent` is your `tiny_agent`
constructor. `agent.run` is your loop. Because you built the forty-line version,
you can predict exactly what these three lines do. That predictive power is the
whole benefit of having built it first.

### On Semantic Kernel (the foundation layer)

Semantic Kernel survived the merger as the base of Agent Framework, so its
concepts are still live. Same story: a service, an agent, a run.

In [ ]:
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from openai import AsyncOpenAI

# SK takes an OpenAI-compatible async client, pointed at the active lane
sk_client = AsyncOpenAI(api_key=api_key, base_url=base_url)
service = OpenAIChatCompletion(ai_model_id=model, async_client=sk_client)

sk_agent = ChatCompletionAgent(
    service=service,
    name="assistant",
    instructions="You are a concise assistant. Answer in one sentence.",
)
sk_response = await sk_agent.get_response(messages="What is an AI agent?")
print("SEMANTIC KERNEL:", sk_response)


Three routes, one behaviour. The frameworks did not invent anything you did
not already build. They packaged it, hardened it, and added production features
you have not needed yet.

## The trap that catches everyone

In [ ]:
from cse476.frameworks import ImportTrap

trap = ImportTrap()
print("symptom:    ", trap.symptom)
print("first check:", trap.first_check)
print()
for line in trap.the_confusion:
    print("  ", line)

Run `pip show agent-framework` and `pip show autogen` in your terminal. Read
what each is. Several packages have near-identical names, and installing the
wrong one is the most common reason a beginner's imports do not match the docs.
Knowing this signals real experience in an interview.

## Your turn

**1. Prove all three.** Run every cell above and confirm the hand-built, Agent
Framework, and Semantic Kernel agents all answer the same question. Same
behaviour, three ways.

**2. Run `pip show`.** On `agent-framework` and `autogen`. Write one line on what
each is, so you can explain the trap to someone else.

**3. Justify a choice.** For your capstone, in three sentences, name the framework
you would use and why, naming at least one you rejected. "Agent Framework because
the course used it" is the weak answer. Justify against a real constraint: your
ecosystem, your team's existing stack, your need for support or openness.

In [ ]:
# your work here
